##RAG Pipelines - Data Ingestion to Vector DB Pipeline


In [16]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [17]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all Pdf files in a directory"""
    all_documents= []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)}")

    for pdf_file in pdf_files:
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTOtal documents loaded: {len(documents)}")

    return all_documents

    #Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

            


Found 1
 Loaded 1 pages

TOtal documents loaded: 1


In [18]:
all_pdf_documents


[Document(metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-06T20:30:46+00:00', 'source': '..\\data\\pdf\\prashasth4.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'prashasth4.pdf', 'file_type': 'pdf'}, page_content='Prashasth Singh \uf0e0 prashasth068@gmail.com \uf08e\nF ull-stack Developer \uf095 +91-7268869676\n\uf0ac Portfolio \uf08e | \uf08c LinkedIn \uf08e | \uf09b GitHub \uf08e \uf041 Lucknow, UP\nEducation\n• Institute of Engineering and T echnology , Lucknow Lucknow, U.P., India\nB. Tech. - Computer Science and Engineering | CGPA: 8.1/10 Nov 2023 - Jun 2027\n• Bhavan’s K D K Vidhya Mandir Renukoot Sonebhadra, U.P., India\nClass XII - Physics, Chemistry, Math, Computer Science | 90.8% Apr 2021 - May 2022\nAchievements\n• LeetCode \uf08e: Knight | Highest Rating: 2039 – Global Rank 245 in BiWeekly Contest 152 (Top 0.8% globally)\n• Codeforces \uf08e: Pupil | Highest Rating: 1282 – Global Rank 2921 in Ro

In [19]:
#Text splitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}..")
        print(f"MetaData: {split_docs[0].metadata}")

    return split_docs

    


In [20]:
chunks = split_documents(all_pdf_documents)

split 1 documents into 4 chunks

Example chunk:
Content: Prashasth Singh  prashasth068@gmail.com 
F ull-stack Developer  +91-7268869676
 Portfolio  |  LinkedIn  |  GitHub   Lucknow, UP
Education
• Institute of Engineering and T echnology , Lucknow..
MetaData: {'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-06T20:30:46+00:00', 'source': '..\\data\\pdf\\prashasth4.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'prashasth4.pdf', 'file_type': 'pdf'}
